# Обнаружение аномалий
## Датасет: banks

## 1. Загрузка данных

In [26]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, jaccard_score 

#указываем кодировку cp1251, так как файл содержит кириллицу и сохранен в windows-кодировке
df = pd.read_csv('banks.txt', sep=',', encoding='cp1251')

#просмотр первых строк для проверки
print("Первые строки датасета:")
print(df.head())

#выбираем числовые колонки для анализа
#исключаем название банка, так как это категориальный признак
numeric_cols = ['Assents', 'OwnCapital', 'IndFunds', 'NBSLoans', 'IndLoans']

#проверяем наличие пропусков и заполняем их нулями
df[numeric_cols] = df[numeric_cols].fillna(0)

Первые строки датасета:
               Bank  Assents  OwnCapital  IndFunds  NBSLoans  IndLoans
0        «Авангард»   122109       20440     35443     32728      3319
1           «Аверс»   110741       24410     34918     13613      4924
2           «Агора»     1114         356       274       351       206
3  «Агропромкредит»    18774        2332     12047      6484       903
4         «Агророс»     7917        1157      3564      1909       492


## 2. Подготовка данных для LOF (novelty=false)

In [27]:
#масштабируем данные, так как lof чувствителен к масштабу признаков
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df[numeric_cols])

#инициализируем lof с настройками по умолчанию
#novelty=false означает режим обнаружения аномалий внутри обучающей выборки
lof_default = LocalOutlierFactor(novelty=False)

#обучаем модель и предсказываем аномалии
#fit_predict возвращает массив: 1 для нормальных точек, -1 для аномалий
predictions_default = lof_default.fit_predict(data_scaled)

#добавляем предсказания в датафрейм
df['lof_outlier_default'] = predictions_default

### Вывод по этапу
Данные масштабированы с помощью StandardScaler.
Алгоритм LOF запущен в режиме novelty=False - это означает, что он обучается и предсказывает аномалии на одном и том же наборе данных (обучающая выборка = тестовая).
Результат сохранён в новую колонку lof_outlier_default: 1 - норма, -1 - аномалия.

## 3. Подсчет числа аномалий

In [28]:
#считаем количество точек, помеченных как -1
n_anomalies_default = (predictions_default == -1).sum()
print(f'\nЧисло аномалий (LOF, novelty=false): {n_anomalies_default}')


Число аномалий (LOF, novelty=false): 77


### Вывод по этапу
Алгоритм LOF выявил 77 аномальных банков из общего числа (около 300+ банков в датасете).
Это примерно 25–30% от всей выборки - довольно высокий процент, что может указывать на то, что многие банки имеют нестандартное соотношение финансовых показателей (например, очень малый капитал при больших активах, или отрицательный капитал).

## 4. Фильтрация аномалий и повторный запуск в режиме novelty=true

In [29]:
#отделяем "нормальные" данные (inliers) для обучения модели новизны
df_inliers = df[df['lof_outlier_default'] == 1].copy()

#масштабируем только нормальные данные тем же скалером
data_inliers_scaled = scaler.transform(df_inliers[numeric_cols])

#инициализируем lof в режиме обнаружения новизны
lof_novelty = LocalOutlierFactor(novelty=True)

#обучаем модель только на нормальных данных
lof_novelty.fit(data_inliers_scaled)

#теперь оценим те самые аномалии, которые мы отфильтровали ранее
df_outliers = df[df['lof_outlier_default'] == -1].copy()

#масштабируем аномалии тем же скалером
data_outliers_scaled = scaler.transform(df_outliers[numeric_cols])

#предсказываем статус для отфильтрованных аномалий
#если модель считает их новизной (аномалией), вернет -1
predictions_novelty = lof_novelty.predict(data_outliers_scaled)

#считаем число совпадений
#сколько из исходных аномалий модель новизны также признала аномалиями (-1)
n_matches = (predictions_novelty == -1).sum()
print(f'Число совпадений (из отфильтрованных аномалий): {n_matches}')

Число совпадений (из отфильтрованных аномалий): 77


### Вывод по этапу
Мы взяли только "нормальные" банки (те, которые LOF не посчитал аномалиями) и обучили на них модель LOF в режиме novelty=True. Теперь эта модель умеет определять, является ли новый объект "новинкой" (аномалией) относительно ранее виденных нормальных данных.
Затем мы проверили, сколько из тех самых 77 аномалий, найденных на первом этапе, эта новая модель также признает аномалиями.
Результат: 77 совпадений.
1. Это означает, что все 77 банков, которые LOF изначально посчитал аномалиями, действительно являются выбросами даже с точки зрения модели, обученной только на "нормальных" банках.
2. Модель устойчива: она не просто помечает случайные точки, а выделяет объекты, которые радикально отличаются от основной массы.

## 5. Решение задачи методом Isolation Forest

In [30]:
#инициализируем isolation forest
#contamination='auto' позволяет алгоритму самому оценить долю аномалий
iso_forest = IsolationForest(contamination='auto', random_state=42)

#обучаем и предсказываем на всех данных
predictions_if = iso_forest.fit_predict(data_scaled)

#добавляем результаты в датафрейм
df['if_outlier'] = predictions_if

#считаем число аномалий по isolation forest
n_anomalies_if = (predictions_if == -1).sum()
print(f'Число аномалий (Isolation Forest): {n_anomalies_if}')

#вывод итоговой таблицы с флагами аномалий
print("\nИтоги по первым 10 банкам:")
print(df[['Bank', 'lof_outlier_default', 'if_outlier']].head(10))

Число аномалий (Isolation Forest): 30

Итоги по первым 10 банкам:
                Bank  lof_outlier_default  if_outlier
0         «Авангард»                    1           1
1            «Аверс»                    1           1
2            «Агора»                    1           1
3   «Агропромкредит»                    1           1
4          «Агророс»                    1           1
5          «Ак Барс»                    1          -1
6         «Акрополь»                    1           1
7           «Акцепт»                    1           1
8  «Александровский»                    1           1
9     «Альба Альянс»                    1           1


### Вывод по этапу
Метод Isolation Forest нашёл 30 аномальных банков.
Это значительно меньше, чем у LOF (77 и 30).

## 6. Расчет метрик

In [32]:
#1. силуэтный коэффициент для lof
#берем только те точки, которые не являются шумом, если бы мы использовали кластеризацию
#но для outlier detection можно оценить качество разделения на 2 группы (норма/аномалия)
#silhouette score требует минимум 2 кластера и не любит дисбаланс
#используем метки lof для оценки
try:
    #преобразуем метки -1 и 1 в 0 и 1 для удобства
    #используем predictions_default, так как именно так названа переменная выше
    labels_lof_binary = (predictions_default == -1).astype(int)
    
    #если аномалий слишком мало или нет, силиуэт может выдать ошибку
    if len(np.unique(labels_lof_binary)) > 1:
        sil_score_lof = silhouette_score(data_scaled, labels_lof_binary)
        print(f'Silhouette score (LOF разделение): {sil_score_lof:.4f}')
    else:
        print('Silhouette score не рассчитан (нет разделения на классы)')
except Exception as e:
    print(f'Ошибка расчета Silhouette score: {e}')

#2. коэффициент жаккарда для множеств аномалий
#создаем бинарные векторы: 1 если аномалия, 0 если норма
binary_lof = (predictions_default == -1).astype(int)
binary_if = (predictions_if == -1).astype(int)

#jaccard_score рассчитывает пересечение/объединение
jac_score = jaccard_score(binary_lof, binary_if)
print(f'Jaccard similarity (пересечение аномалий): {jac_score:.4f}')

Silhouette score (LOF разделение): 0.6016
Jaccard similarity (пересечение аномалий): 0.3210


### Выбор метрик (Silhouette Score, Jaccard) обусловлен спецификой задачи обучения без учителя и отсутствием истинных меток.

1. Почему Silhouette Score?

Это стандартная метрика для оценки качества кластеризации или разделения данных. Она измеряет, насколько хорошо объекты отделены друг от друга.
В нашем случае LOF и Isolation Forest по сути делят данные на два «кластера»: нормальные банки и аномальные банки. Silhouette Score показывает, насколько четко проведена эта граница.
а) Если score высокий (близкий к 1), значит, аномалии действительно выглядят как отдельная, плотная группа, отличная от остальных.
б) Если score низкий (около 0), значит, граница размыта, и модель, возможно, просто случайно выдернула какие-то точки.

2. Почему Jaccard similarity?

Так как у нас нет эталона, лучшим способом проверки надежности является сравнение двух разных алгоритмов (LOF и Isolation Forest). LOF ищет аномалии, основываясь на локальной плотности (кто рядом?), а Isolation Forest - на легкости изоляции в дереве решений (как далеко от центра?). Это принципиально разные подходы.
а) Если оба метода указывают на одни и те же банки (высокий Jaccard), значит, эти банки являются очевидными выбросами (например, у них гигантские активы при нулевом капитале).
б) Если совпадений мало, значит, каждый алгоритм видит свои, специфические типы аномалий.